# Data manipulation

In this notebook we will load and explore the data and try to answer two basic questions:

1. Detail the experimental conditions
2. Visualise Cell Painting processed data at different levels of the data 


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import umap
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.colors import ListedColormap, BoundaryNorm, Normalize
if not hasattr(np, "float"): # To avoir numpy/cytomyner_eval attribute error
    np.float = float
if not hasattr(np, "int"):
    np.int = int
if not hasattr(np, "bool"):
    np.bool = bool
if not hasattr(np, "str"):
    np.str = str
from copairs import map
from copairs.matching import assign_reference_index

In [ ]:
# Read in normalised data at the replicate level
df_level4b = pd.read_csv('../outputs/df_level4b.csv')

# Get lists of metadata features and CellProfiler features
meta_features = df_level4b.columns[df_level4b.columns.str.startswith("Metadata_")].to_list()
features = df_level4b.drop(meta_features, axis="columns").columns.tolist()

## Detect morphological differences between treated and control samples

Mean average precision is a metric that can calculate differences between whole profiles. It is implemented in the copairs package. For more details, see [this tutorial](https://github.com/cytomining/copairs/blob/main/docs/examples/phenotypic_activity.ipynb) on using mAP for detecting differences between treatments and controls, or [the paper](https://www.nature.com/articles/s41467-025-60306-2) that demonstrates how mAP can be used for cell profiling.

In [ ]:
reference_col = "Metadata_reference_index"

# Make a column to 
df_activity = assign_reference_index(
                            df_level4b,
                            "Metadata_Name == 'DMSO'",  # condition to get reference profiles (neg controls)
                            reference_col=reference_col,
                            default_value=-1,
                            )

# positive pairs are replicates of the same treatment
pos_sameby = ["Metadata_JCP2022", reference_col]
pos_diffby = []

neg_sameby = []
# negative pairs are replicates of different treatments
neg_diffby = ["Metadata_JCP2022", reference_col]

metadata = df_activity.filter(regex="^Metadata")
profiles = df_activity[features].values

activity_ap = map.average_precision(
metadata, profiles, pos_sameby, pos_diffby, neg_sameby, neg_diffby
)
activity_ap_df = activity_ap.query("Metadata_Name != 'DMSO'")  # remove DMSO

activity_map_df = map.mean_average_precision(
                                    activity_ap_df, pos_sameby, null_size=1000000, threshold=0.05, seed=0
                                    )

# Write out results to outputs
activity_ap_df.to_csv('../outputs/average_precision.csv', index=False)
activity_map_df.to_csv('../outputs/mean_average_precision.csv', index=False)

**Exercise**
- Which five compounds have the most significant difference compared to the negative controls? Do some quick Google / ChatGPT searches. What do these compounds do to human cells?
- How many compounds have a detectable difference compared to the controls according to mAP?
- Some of the compounds are as positive or negative controls. What percentage of these have detectable phenotypic differences?
- mAP looks for differences between entire profiles. In bioinformatics, it's more common to look for differences between individual features (ie. differentially expressesd genes). Pick one of the compounds with a statistically significant mAP score, and identify the CellProfiler features that have the most significant differences between treatment and control samples. How might you construct these statistical tests to account for the plate covariate?

In [ ]:
# Bonus: make a plot to visualise the compounds
df_map = pd.read_csv("../outputs/mean_average_precision.csv")
df_map["-log10(p-value)"] = -df_map["corrected_p_value"].apply(np.log10)

active_ratio = df_map.below_corrected_p.mean()

plt.scatter(
    data=df_map,
    x="mean_average_precision",
    y="-log10(p-value)",
    c="below_corrected_p",
    cmap="tab10",
    s=10,
)
plt.title("Phenotypic activity assesement")
plt.xlabel("mAP")
plt.ylabel("-log10(p-value)")
plt.axhline(-np.log10(0.05), color="black", linestyle="--")
plt.text(
    0.65,
    1.5,
    f"Phenotypically active = {100 * active_ratio:.2f}%",
    va="center",
    ha="left",
)
plt.show()

## Cluster compound by mode-of-action

For this section, we will use the consensus profiles so that we just have one profile per compound. We will use various dimensionality reduction and clustering methods to identify groups of compounds that have similar effects in cells.

In [ ]:
# Load data
df_level5 = pd.read_csv("../outputs/df_level5.csv")

# Define important arguments
meta_features = df_level5.columns[df_level5.columns.str.startswith("Metadata_")].to_list()
features = df_level5.drop(meta_features, axis="columns").columns.tolist()

# Load mAP
df_map = pd.read_csv("../outputs/mean_average_precision.csv")
df_map["-log10(p-value)"] = -df_map["corrected_p_value"].apply(np.log10)

# merge dt_level5 and map
df_level5_map = pd.merge(
    df_level5,
    df_map,
    on=["Metadata_JCP2022"],
    how="left"   # or "left", "right", "outer" depending on what you need
)

### Heatmap

We will make a heatmap of profiles and use clustering to identify groups of samples and features with similar patterns. 

In [ ]:
# --- Extract the feature matrix ---
feature_data = df_level5_map[features]

# --- Extract channel info from feature names ---
channels = [f.split("_")[0] if "_" in f else "Unknown" for f in features]

# --- Channel color palette ---
unique_channels = sorted(set(channels))
palette = sns.color_palette("Set2", len(unique_channels))
channel_colors = dict(zip(unique_channels, palette))
col_colors = pd.Series(channels, index=features).map(channel_colors)

# --- Clip data range ---
feature_data_clipped = feature_data.clip(lower=-5, upper=5)

# --- Left annotation: Perturbation type ---
pert_types = df_level5_map["Metadata_pert_type"].astype(str)
unique_pert_types = sorted(pert_types.unique())
palette_perts = sns.color_palette("Pastel1", len(unique_pert_types))
pert_colors = dict(zip(unique_pert_types, palette_perts))
row_colors = pert_types.map(pert_colors)

# --- Right annotation: -log10(p-value) ---
pvals = df_level5_map["-log10(p-value)"]
norm = Normalize(vmin=pvals.min(), vmax=pvals.max())
cmap_pval = plt.cm.get_cmap("YlOrRd")  # yellow to red color gradient
pval_colors = [cmap_pval(norm(v)) for v in pvals]

# Combine both left and right row annotations
# Seaborn only supports ONE row color DataFrame, so we merge them side by side
row_colors_combined = pd.DataFrame({
    "Perturbation": row_colors,
    "-log10(p-value)": pval_colors
}, index=df_level5_map.index)

# --- Create clustermap ---
sns.set(style="white")
g = sns.clustermap(
    feature_data_clipped,
    cmap="vlag",
    col_colors=col_colors,
    row_colors=row_colors_combined,
    xticklabels=False,
    yticklabels=False,
    figsize=(13, 8),
    center=0,
    vmin=-5,
    vmax=5,
    row_cluster=True,
    col_cluster=False,
    method='ward'
)

# --- Add legends ---
# Channels legend (top)
for label in unique_channels:
    g.ax_col_dendrogram.bar(0, 0, color=channel_colors[label], label=label, linewidth=0)
g.ax_col_dendrogram.legend(
    title="Channels",
    loc="center",
    ncol=len(unique_channels),
    bbox_to_anchor=(0.5, 1.1)
)

# Perturbation type legend (left)
for label in unique_pert_types:
    g.ax_row_dendrogram.bar(0, 0, color=pert_colors[label], label=label, linewidth=0)
g.ax_row_dendrogram.legend(
    title="Perturbation Type",
    loc="center left",
    bbox_to_anchor=(-0.25, 0.5)
)

# --- Add colorbar for -log10(p-value) ---
sm = plt.cm.ScalarMappable(cmap=cmap_pval, norm=norm)
sm.set_array([])

# Instead of attaching directly to ax_heatmap,
# use the figure's add_axes() to manually position it
# [left, bottom, width, height] — relative to figure size
cbar_ax = g.fig.add_axes([0.93, 0.2, 0.02, 0.6])
cbar = g.fig.colorbar(sm, cax=cbar_ax)
cbar.set_label("-log10(p-value)", rotation=270, labelpad=15)

# --- Adjust layout ---
g.fig.subplots_adjust(right=0.9)  # leave room for colorbar

plt.show()

**Exercise**

- The heatmap above sorts features according to whether they came from the cell, cytoplasm, or nucleus. Adjust the code to allow the features to cluster in addition to the samples.
- Add a second layer of annotations to the columns that indicates which channel the features come from. Do features tend to cluster by channel or by compartment?
- Turn off feature (column) clustering and sort the CellProfiler features by channel before remaking the plot. Are there some compounds that impact some channels more than others?

## Dimensionality reduction with UMAP

UMAP is a method to find low dimension representations of samples that preserve local distances between samples in high dimensions. We can examine UMAP scores plots to look for groups of compounds with similar profiles.

In [ ]:
# --- Prepare data ---
# Clip or standardize features to improve embedding quality
from sklearn.preprocessing import StandardScaler
X_scaled = StandardScaler().fit_transform(feature_data)

# --- Run UMAP ---
reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    metric="euclidean",
    random_state=42
)
embedding = reducer.fit_transform(X_scaled)

In [ ]:
# --- Prepare coloring ---
pvals = df_level5_map["-log10(p-value)"]
norm = plt.Normalize(vmin=pvals.min(), vmax=pvals.max())
cmap = plt.cm.YlOrRd

# --- Plot UMAP ---
plt.figure(figsize=(10, 8))
sc = plt.scatter(
    embedding[:, 0],
    embedding[:, 1],
    c=pvals,
    cmap=cmap,
    norm=norm,
    s=50,
    edgecolor="none",
    alpha=0.9
)
plt.title("UMAP Projection Colored by -log10(p-value)", fontsize=14)
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")

# --- Add colorbar ---
cbar = plt.colorbar(sc)
cbar.set_label("-log10(p-value)", rotation=270, labelpad=15)

# --- Highlight and label DMSO points ---
dmso_mask = df_level5_map["Metadata_Name"] == "DMSO"
plt.scatter(
    embedding[dmso_mask, 0],
    embedding[dmso_mask, 1],
    color="none",
    s=80,
    edgecolor="black",
    linewidth=1.5,
    label="DMSO"
)

# --- Label samples with non-null Metadata_pert_type ---
valid_mask = df_level5_map["Metadata_pert_type"].notna()
for i in np.where(valid_mask)[0]:
    label = str(df_level5_map.loc[i, "Metadata_Name"]) + '(' + str(df_level5_map.loc[i, "Metadata_pert_type"]) + ')'
    plt.text(
        embedding[i, 0],
        embedding[i, 1],
        label,
        fontsize=8,
        color="black",
        ha="center",
        va="center",
        weight="semibold",
        alpha=0.8,
        bbox=dict(facecolor="white", alpha=0.5, edgecolor="none", pad=1)
    )

# --- Add legend ---
plt.legend(frameon=False, loc="best")

plt.tight_layout()
plt.show()

**Exercise**
- The UMAP only has a few text labels because it would be too crowded if we used all of them. Try and implement a UMAP plot with an interactive library like plotly such that you can hover over a point to reveal the name and p-value.